In [54]:
from dataclasses import dataclass
from datetime import date, datetime
from typing import Optional, List
import requests
from datetime import datetime
import json
import math

In [2]:
@dataclass
class Breach:
    name: str
    domain: str
    breach_date: date
    added_date: Optional[date]
    data_classes: List[str]
    pwn_count: int
    is_verified: bool
    is_MalWare: bool    # added to calc risk
    is_StealerLog: bool # added to calc risk

In [3]:
@dataclass
class Paste:
    id: str
    source: str
    date: Optional[datetime]
    email_count: Optional[int]

In [4]:
@dataclass
class BreachSummary:
    breaches: List[Breach]
    risk_score: float

In [53]:
def parse_breach_list(json_data: list[dict]) -> List[Breach]:
    breaches = []
    for item in json_data:
        bd = item.get('BreachDate')
        ad = item.get('AddedDate')
        breach_date = datetime.strptime(bd, '%Y-%m-%d').date() if bd else None
        added_date = datetime.strptime(ad, '%Y-%m-%dT%H:%M:%SZ').date() if ad else None

        breach = Breach(
            name = item.get('Name'),
            domain = item.get('Domain'),
            breach_date = breach_date,
            added_date = added_date,
            data_classes = item.get('DataClasses',[]),
            pwn_count = item.get('PwnCount',0),
            is_verified = item.get('IsVerified',False),
            is_MalWare = item.get('IsMalware',False),           # added to calc risk
            is_StealerLog = item.get('IsStealerLog',False)      # added to calc risk
        )
        breaches.append(breach)
    return breaches

In [6]:
def parse_paste_list(json_data: list[dict]) -> List[Paste]:
    pastes = []
    for item in json_data:
        paste = Paste(
            id = item.get('Id'),
            source = item.get('Source'),
            date = item.get('Date'),
            email_count = item.get('EmailCount'),
        )
        pastes.append(paste)
    return pastes

In [41]:
def group_breaches_by_year(breaches: List[Breach]) -> dict[int, List[Breach]]:
    result = {}
    for breach in breaches:
        result.setdefault(breach.breach_date.year, []).append(breach)
    return result

In [8]:
def group_breaches_by_domain(breaches: List[Breach]) -> dict[str, List[Breach]]:
    result = {}
    for breach in breaches:
        domain = breach.domain
        result.setdefault(domain, []).append(breach)
    return result

In [55]:
# calc risk score:
#   - there is no official calc method
#   - normalization is difficult (0-100-model)
#   - idea: score weight on data_classes, which is the label of leaked information https://haveibeenpwned.com/api/v3/dataclasses
#   - there are 150 entries in dataclasses, typical labels are chosen manually to calc.

def calculate_risk_score(breaches: List[Breach]) -> float:
    with open("data_class_risk_map.json","r",encoding="utf-8") as f:
        risk_factors = json.load(f)

    high = risk_factors['high']
    medium = risk_factors['medium']
    low = risk_factors['low']
    weight = risk_factors['weight']
    event_tag = risk_factors['event_tag']

    risk_score = 0

    for breach in breaches:
        for entry in breach.data_classes:
            if entry in high:
                risk_score += weight['high']
            elif entry in medium:
                risk_score += weight['medium']
            elif entry in low:
                risk_score += weight['low']
            else:
                risk_score += 1

        if breach.is_MalWare:
            risk_score += event_tag['malware']
        if breach.is_StealerLog:
            risk_score += event_tag['stealerLog']
        if breach.is_verified:
            risk_score += event_tag['verified']

        risk_score = int(risk_score + math.log10(breach.pwn_count + 1) * 2)

    return risk_score

In [61]:
def build_breach_summary(breaches: List[Breach]) -> BreachSummary:
    score = calculate_risk_score(breaches)

    return BreachSummary(
        breaches = breaches,
        risk_score = score
    )

In [62]:
if __name__ == "__main__":

    test_email = "account-exists@hibp-integration-tests.com"
    print(f"Testing connection for email: {test_email}")

    # --- Fetch breaches ---
    breaches = fetch_breaches(test_email)

    if not breaches:
        print("No breaches found for this email.")
        exit()

    print(f"\n✔ Found {len(breaches)} breaches.\n")

    # --- Group by year ---
    print("===== Group by Year =====")
    by_year = group_breaches_by_year(breaches)
    for year, items in sorted(by_year.items()):
        print(f"{year}: {len(items)} breaches")

    # --- Group by domain ---
    print("\n===== Group by Domain =====")
    by_domain = group_breaches_by_domain(breaches)
    for domain, items in by_domain.items():
        print(f"{domain}: {len(items)} breaches")
    print()

    # --- Detailed breach info ---
    print("===== Breach Details =====")
    for b in breaches:
        print(f"\n{b.name}")
        print(f"   Domain: {b.domain}")
        print(f"   Date:   {b.breach_date}")
        print(f"   Verified: {b.is_verified}")
        print(f"   Malware:  {b.is_MalWare}")
        print(f"   StealerLog: {b.is_StealerLog}")
        print(f"   Classes: {', '.join(b.data_classes[:5])}...")
        print(f"   PwnCount: {b.pwn_count}")

    # --- Build summary (risk score) ---
    print("\n===== Risk Summary =====")
    summary = build_breach_summary(breaches)

    print(f"Total Risk Score: {summary.risk_score} from event: {summary.breaches}")

Testing connection for email: account-exists@hibp-integration-tests.com
Testing connection: account-exists@hibp-integration-tests.com 
Fetched: 1 breaches。

✔ Found 1 breaches.

===== Group by Year =====
2013: 1 breaches

===== Group by Domain =====
adobe.com: 1 breaches

===== Breach Details =====

Adobe
   Domain: adobe.com
   Date:   2013-10-04
   Verified: True
   Malware:  False
   StealerLog: False
   Classes: Email addresses, Password hints, Passwords, Usernames...
   PwnCount: 152445165

===== Risk Summary =====
Total Risk Score: 101 from event: [Breach(name='Adobe', domain='adobe.com', breach_date=datetime.date(2013, 10, 4), added_date=datetime.date(2013, 12, 4), data_classes=['Email addresses', 'Password hints', 'Passwords', 'Usernames'], pwn_count=152445165, is_verified=True, is_MalWare=False, is_StealerLog=False)]


In [56]:
def fetch_breaches(email: str) -> List[Breach]:
    url = f"https://haveibeenpwned.com/api/v3/breachedaccount/{email}?truncateResponse=false"

    headers = {
        "hibp-api-key": "00000000000000000000000000000000", # Test API Key
        "user-agent": "TestScript-v1",
        "content-type": "application/json"
    }

    print(f"Testing connection: {email} ")

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            json_data = response.json()
            print(f"Fetched: {len(json_data)} breaches。")
            return parse_breach_list(json_data)

        elif response.status_code == 404:
            print("404")
            return []
        elif response.status_code == 401:
            print("401")
        else:
            print(f"Error: {response.status_code} - {response.text}")

    except Exception as e:
        print(f"Error: {e}")

    return []